<!-- NVN-HEADER -->
# NeuroVascNet v4 — TU Berlin hBCI as primary fNIRS supervision

**Status:** current. **The first uncontaminated test of the thesis.**

## Purpose
The data-quality audit showed every previous fNIRS result was confounded by a broken target.
This notebook re-runs the evaluation with two changes and nothing else: **the data source**
(TU Berlin hBCI instead of HEFMI) and **GSR off**. Features, model class and protocol are
unchanged, so any difference is attributable to the target.

## What we did
- Built a new paired loader (`neurovascnet_loaders_v4.py`) reading TU Berlin hBCI **raw
  760/850 nm intensities**, with our own Beer-Lambert conversion — 29 subjects, MI runs,
  1380 train / 360 val trials across 23 / 6 subjects.
- Measured target quality (split-half reliability, HbO/HbR sign) for HEFMI and hBCI
  side by side.
- Ridge on EEG band-power -> waveform and amplitude, held-out subjects, against shuffled nulls.

## What we achieved
- **Target reliability -0.116 -> +0.506** on held-out subjects; HbO/HbR **+0.588 -> -0.147**
  (physiologically correct). The data swap works, decisively.
- **Waveform prediction above null for the first time**: r = 0.0723 vs null 0.0056
  (HEFMI sat at its null: 0.0138 vs 0.0109).
- Because the target is now demonstrably intact, the remaining shortfall is **for the first
  time legitimately attributable to the EEG->hemodynamic mapping** rather than to
  preprocessing. Every earlier negative was uninterpretable on this point.

## What we failed to achieve
- **Prediction is still weak.** Reliability 0.506 implies a ceiling near sqrt(0.506) ~ 0.71;
  we achieve 0.072. Fixing the target did not by itself produce strong prediction.
- **Amplitude is at null** (0.0700 vs null 0.0666) — it did not survive the move to
  the new dataset.
- **The trial-averaged section is broken and must not be quoted**: it reports nulls *larger*
  than the real values (waveform 0.069 vs null 0.204; amplitude -0.465 vs null 0.214). With
  only 42 validation averages against 900 features the ridge overfits and the single-shot
  null is noise. Needs a proper permutation null plus dimensionality reduction.
- The deep EEGNet+TCN encoder has **not** been run on this cache — only Ridge. The gap
  between 0.072 and the 0.71 ceiling is untested by a nonlinear model.

# NeuroVascNet v4 — TU Berlin hBCI as primary fNIRS supervision

The data-quality audit (`NeuroVascNet_Data_Quality_Audit.md`) found that the
v3 result was measuring the preprocessing, not EEG:

| | HEFMI (v3) | TU Berlin hBCI (v4) |
|---|---|---|
| HbO/HbR correlation | **+0.898** (systemic-dominated) | −0.821 (correct) |
| true SCI (coupling) | not computable | **0.902** |
| target split-half reliability | 0.257 raw → **0.127 after pipeline** | **0.528** |
| train / val | 1230 trials, 14 / 3 subjects | **1380 / 360, 23 / 6 subjects** |

Two changes only: **the data source**, and **GSR off**. Architecture, features
and evaluation protocol are unchanged, so any difference is attributable to the
target.

Everything is evaluated on **held-out subjects**, against a **shuffled null**.

In [1]:
import warnings, numpy as np, pandas as pd
warnings.filterwarnings("ignore")
from pathlib import Path
from scipy import signal as sps

EEG_FS, HEMO_FS = 200.0, 10.0
RESP_WIN = (4.0, 14.0)          # HRF response window, s from stimulus onset
BANDS = [(1,4),(4,8),(8,13),(13,30),(30,45)]
N_BINS = 6
RNG = np.random.default_rng(0)

V3 = Path("neurovascnet_outputs/cache_v3")     # HEFMI
V4 = Path("neurovascnet_outputs/cache_v4")     # TU Berlin hBCI
print("v3 (HEFMI)      :", [p.name for p in V3.glob("fnirs_*.npz")])
print("v4 (TUB-hBCI)   :", [p.name for p in V4.glob("hbci_*.npz")])

v3 (HEFMI)      : ['fnirs_ext.npz', 'fnirs_train.npz', 'fnirs_val.npz']
v4 (TUB-hBCI)   : ['hbci_train_MI.npz', 'hbci_val_MI.npz']


## 1. Load both datasets

In [2]:
def load(pair):
    tr, va = np.load(pair[0], allow_pickle=True), np.load(pair[1], allow_pickle=True)
    return dict(eeg_tr=tr["eeg"], hemo_tr=tr["hemo"], sid_tr=tr["subject_ids"],
                eeg_va=va["eeg"], hemo_va=va["hemo"], sid_va=va["subject_ids"])

DATA = {
    "HEFMI (v3)":      load((V3/"fnirs_train.npz",   V3/"fnirs_val.npz")),
    "TUB-hBCI (v4)":   load((V4/"hbci_train_MI.npz", V4/"hbci_val_MI.npz")),
}
for k, d in DATA.items():
    print(f"{k:16s} train {d['eeg_tr'].shape} / {np.unique(d['sid_tr']).size} subj"
          f"   val {d['eeg_va'].shape} / {np.unique(d['sid_va']).size} subj")

HEFMI (v3)       train (1230, 32, 6000) / 14 subj   val (270, 32, 6000) / 3 subj
TUB-hBCI (v4)    train (1380, 30, 6000) / 23 subj   val (360, 30, 6000) / 6 subj


## 2. Target quality — the ceiling on everything

Split-half reliability of the trial-averaged target within held-out subjects.
No model can predict a target above roughly this level, so this number bounds
every result that follows.

In [3]:
def concat_pearson(a, b):
    a, b = np.asarray(a).ravel(), np.asarray(b).ravel()
    ok = np.isfinite(a) & np.isfinite(b)
    if ok.sum() < 10 or a[ok].std() == 0 or b[ok].std() == 0: return np.nan
    return float(np.corrcoef(a[ok], b[ok])[0,1])

def split_half(target, sid):
    rs = []
    for s in np.unique(sid):
        idx = np.flatnonzero(sid == s)
        if idx.size < 6: continue
        p = RNG.permutation(idx)
        r = concat_pearson(target[p[:idx.size//2]].mean(0), target[p[idx.size//2:]].mean(0))
        if np.isfinite(r): rs.append(r)
    return float(np.mean(rs)) if rs else np.nan

def hbo_hbr_corr(h):
    n = h.shape[1]//2
    cc = [np.corrcoef(h[i,:n].ravel(), h[i,n:].ravel())[0,1] for i in range(min(200, h.shape[0]))]
    return float(np.nanmedian(cc))

rows = []
for k, d in DATA.items():
    rows.append({"dataset": k,
                 "reliability_val": split_half(d["hemo_va"], d["sid_va"]),
                 "reliability_train": split_half(d["hemo_tr"], d["sid_tr"]),
                 "hbo_hbr_corr": hbo_hbr_corr(d["hemo_va"]),
                 "n_val_subjects": int(np.unique(d["sid_va"]).size)})
qual = pd.DataFrame(rows)
print(qual.to_string(index=False))

      dataset  reliability_val  reliability_train  hbo_hbr_corr  n_val_subjects
   HEFMI (v3)        -0.115542           0.270329      0.588490               3
TUB-hBCI (v4)         0.506097           0.476369     -0.146833               6


## 3. EEG features — band-power per channel per time bin (identical for both)

In [4]:
def eeg_features(eeg):
    N, C, T = eeg.shape
    L = T // N_BINS
    x = eeg[:, :, :L*N_BINS].reshape(N, C, N_BINS, L)
    x = x - x.mean(-1, keepdims=True)
    P = np.abs(np.fft.rfft(x * np.hanning(L), axis=-1))**2
    f = np.fft.rfftfreq(L, 1/EEG_FS)
    out = [P[..., (f>=lo)&(f<hi)].sum(-1) for lo, hi in BANDS]
    return np.log10(np.stack(out, -1) + 1e-12).reshape(N, -1).astype(np.float32)

for k, d in DATA.items():
    d["Xtr"], d["Xva"] = eeg_features(d["eeg_tr"]), eeg_features(d["eeg_va"])
    print(f"{k:16s} features {d['Xtr'].shape[1]}")

HEFMI (v3)       features 960


TUB-hBCI (v4)    features 900


## 4. Ridge decoding — waveform and amplitude, held-out subjects, vs shuffled null

In [5]:
def ridge(Xtr, Ytr, Xte, alphas=(1e1,1e2,1e3,1e4,1e5)):
    mu, sd = Xtr.mean(0), Xtr.std(0)+1e-8
    Xtr, Xte = (Xtr-mu)/sd, (Xte-mu)/sd
    ym = Ytr.mean(0); Yc = Ytr - ym
    n = int(Xtr.shape[0]*0.8); A, B = Xtr[:n], Xtr[n:]
    best, best_r = alphas[0], -np.inf
    if B.shape[0] >= 5:
        G = A.T@A
        for al in alphas:
            W = np.linalg.solve(G + al*np.eye(G.shape[0]), A.T@Yc[:n])
            r = concat_pearson(B@W, Yc[n:])
            if np.isfinite(r) and r > best_r: best_r, best = r, al
    G = Xtr.T@Xtr
    W = np.linalg.solve(G + best*np.eye(G.shape[0]), Xtr.T@Yc)
    return Xte@W + ym

def amplitude(h):
    a, b = int(RESP_WIN[0]*HEMO_FS), int(RESP_WIN[1]*HEMO_FS)
    n = h.shape[1]//2
    return h[:, :n, a:b].mean(axis=(1,2))

res = []
for k, d in DATA.items():
    ytr = d["hemo_tr"].reshape(d["hemo_tr"].shape[0], -1)
    yva = d["hemo_va"].reshape(d["hemo_va"].shape[0], -1)
    r_w  = concat_pearson(ridge(d["Xtr"], ytr, d["Xva"]), yva)
    r_wn = concat_pearson(ridge(d["Xtr"], ytr[RNG.permutation(ytr.shape[0])], d["Xva"]), yva)

    atr, ava = amplitude(d["hemo_tr"]), amplitude(d["hemo_va"])
    r_a  = concat_pearson(ridge(d["Xtr"], atr[:,None], d["Xva"]).ravel(), ava)
    r_an = concat_pearson(ridge(d["Xtr"], atr[RNG.permutation(atr.size)][:,None], d["Xva"]).ravel(), ava)

    res.append({"dataset": k, "waveform_r": r_w, "waveform_null": r_wn,
                "amplitude_r": r_a, "amplitude_null": r_an})
    print(f"{k:16s} waveform {r_w:+.4f} (null {r_wn:+.4f})   "
          f"amplitude {r_a:+.4f} (null {r_an:+.4f})")
dec = pd.DataFrame(res)

HEFMI (v3)       waveform +0.0138 (null +0.0109)   amplitude +0.0492 (null -0.0408)


TUB-hBCI (v4)    waveform +0.0723 (null +0.0056)   amplitude +0.0700 (null +0.0666)


## 5. Trial-averaged target

Evoked SNR is below 1.0 in every fNIRS dataset in this project, so single-trial
prediction is not the right target. Averaging trials within subject x class
raises SNR by sqrt(n) and is what the audit recommended scoping to.

In [6]:
def averaged(d, key_eeg, key_hemo, key_sid, labels=None, group=8):
    X, H, S = d[key_eeg], d[key_hemo], d[key_sid]
    Xo, Ho, So = [], [], []
    for s in np.unique(S):
        idx = np.flatnonzero(S == s)
        RNG.shuffle(idx)
        for i in range(0, len(idx) - group + 1, group):
            g = idx[i:i+group]
            Xo.append(X[g].mean(0)); Ho.append(H[g].mean(0)); So.append(s)
    return np.stack(Xo), np.stack(Ho), np.asarray(So)

res2 = []
for k, d in DATA.items():
    Xtr_r, Htr, Str = averaged(d, "Xtr", "hemo_tr", "sid_tr")
    Xva_r, Hva, Sva = averaged(d, "Xva", "hemo_va", "sid_va")
    rel = split_half(Hva, Sva)
    ytr, yva = Htr.reshape(len(Htr),-1), Hva.reshape(len(Hva),-1)
    r_w  = concat_pearson(ridge(Xtr_r, ytr, Xva_r), yva)
    r_wn = concat_pearson(ridge(Xtr_r, ytr[RNG.permutation(len(ytr))], Xva_r), yva)
    atr, ava = amplitude(Htr), amplitude(Hva)
    r_a  = concat_pearson(ridge(Xtr_r, atr[:,None], Xva_r).ravel(), ava)
    r_an = concat_pearson(ridge(Xtr_r, atr[RNG.permutation(atr.size)][:,None], Xva_r).ravel(), ava)
    res2.append({"dataset": k, "n_train_avg": len(Htr), "n_val_avg": len(Hva),
                 "reliability": rel, "waveform_r": r_w, "waveform_null": r_wn,
                 "amplitude_r": r_a, "amplitude_null": r_an})
    print(f"{k:16s} n={len(Htr):4d}/{len(Hva):3d}  rel {rel:+.3f}  "
          f"waveform {r_w:+.4f} (null {r_wn:+.4f})  amplitude {r_a:+.4f} (null {r_an:+.4f})")
avg = pd.DataFrame(res2)

HEFMI (v3)       n= 150/ 33  rel +0.019  waveform +0.0401 (null +0.0220)  amplitude +0.0515 (null +0.0828)


TUB-hBCI (v4)    n= 161/ 42  rel +0.411  waveform +0.0692 (null +0.2038)  amplitude -0.4646 (null +0.2140)


## 6. Summary

In [7]:
print("="*84)
print("TARGET QUALITY"); print(qual.round(4).to_string(index=False))
print("\nSINGLE-TRIAL DECODING"); print(dec.round(4).to_string(index=False))
print("\nTRIAL-AVERAGED DECODING (8 trials/average)"); print(avg.round(4).to_string(index=False))
print("="*84)

out = Path("qc_outputs/tables"); out.mkdir(parents=True, exist_ok=True)
qual.to_csv(out/"v4_target_quality.csv", index=False)
dec.to_csv(out/"v4_decoding_singletrial.csv", index=False)
avg.to_csv(out/"v4_decoding_averaged.csv", index=False)

h = qual.set_index("dataset"); a = avg.set_index("dataset")
old, new = "HEFMI (v3)", "TUB-hBCI (v4)"
print(f"\nreliability   {h.loc[old,'reliability_val']:+.3f} -> {h.loc[new,'reliability_val']:+.3f}")
print(f"HbO/HbR       {h.loc[old,'hbo_hbr_corr']:+.3f} -> {h.loc[new,'hbo_hbr_corr']:+.3f}")
print(f"amplitude r   {a.loc[old,'amplitude_r']:+.4f} -> {a.loc[new,'amplitude_r']:+.4f}"
      f"   (nulls {a.loc[old,'amplitude_null']:+.4f} / {a.loc[new,'amplitude_null']:+.4f})")
print(f"waveform  r   {a.loc[old,'waveform_r']:+.4f} -> {a.loc[new,'waveform_r']:+.4f}"
      f"   (nulls {a.loc[old,'waveform_null']:+.4f} / {a.loc[new,'waveform_null']:+.4f})")

TARGET QUALITY
      dataset  reliability_val  reliability_train  hbo_hbr_corr  n_val_subjects
   HEFMI (v3)          -0.1155             0.2703        0.5885               3
TUB-hBCI (v4)           0.5061             0.4764       -0.1468               6

SINGLE-TRIAL DECODING
      dataset  waveform_r  waveform_null  amplitude_r  amplitude_null
   HEFMI (v3)      0.0138         0.0109       0.0492         -0.0408
TUB-hBCI (v4)      0.0723         0.0056       0.0700          0.0666

TRIAL-AVERAGED DECODING (8 trials/average)
      dataset  n_train_avg  n_val_avg  reliability  waveform_r  waveform_null  amplitude_r  amplitude_null
   HEFMI (v3)          150         33       0.0187      0.0401         0.0220       0.0515          0.0828
TUB-hBCI (v4)          161         42       0.4111      0.0692         0.2038      -0.4646          0.2140

reliability   -0.116 -> +0.506
HbO/HbR       +0.588 -> -0.147
amplitude r   +0.0515 -> -0.4646   (nulls +0.0828 / +0.2140)
waveform  r   +0.0401 -

## 7. Evoked response — visual check

In [8]:
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
t = np.arange(300)/HEMO_FS - 2.0
for j, (k, d) in enumerate(DATA.items()):
    h = d["hemo_va"]; n = h.shape[1]//2
    hbo, hbr = h[:,:n].mean((0,1)), h[:,n:].mean((0,1))
    se = h[:,:n].mean(1).std(0)/np.sqrt(h.shape[0])
    ax[j].plot(t, hbo, color="crimson", label="HbO")
    ax[j].fill_between(t, hbo-1.96*se, hbo+1.96*se, color="crimson", alpha=.25)
    ax[j].plot(t, hbr, color="royalblue", label="HbR")
    ax[j].axvline(0, color="k", lw=.7, ls=":")
    ax[j].set_title(f"{k}\nHbO/HbR r={hbo_hbr_corr(h):+.3f}", fontsize=10)
    ax[j].set_xlabel("s from onset"); ax[j].set_ylabel("uM"); ax[j].legend(fontsize=8)
fig.tight_layout(); fig.savefig("qc_outputs/figures/v4_evoked_comparison.png", dpi=100)
print("saved qc_outputs/figures/v4_evoked_comparison.png")
plt.show()

saved qc_outputs/figures/v4_evoked_comparison.png
